In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

In [2]:
train_data = pd.read_csv("train_data.csv")
test_data = pd.read_csv("test_data.csv")
referensi_film = pd.read_csv("referensi_film.csv")  

print("Data berhasil dibaca")

Data berhasil dibaca


In [3]:
cek_data = pd.DataFrame({
    "File": ["train_data", "test_data", "referensi_film"],
    "Jumlah Baris": [
        len(train_data),
        len(test_data),
        len(referensi_film)
    ],
    "Jumlah Kolom": [
        train_data.shape[1],
        test_data.shape[1],
        referensi_film.shape[1]
    ]
})

display(cek_data)

,File,Jumlah Baris,Jumlah Kolom
0,train_data,80419,3
1,test_data,20417,3
2,referensi_film,9742,3


In [4]:
matriks_user_film = train_data.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
)

print("Matriks user-film berhasil dibuat.")
print("Ukuran matriks:", matriks_user_film.shape)

display(matriks_user_film.iloc[:5, :8])

Matriks user-film berhasil dibuat.
Ukuran matriks: (610, 8957)


movieId,1,2,3,4,5,6,7,8
userId,,,,,,,,
1,NaN,NaN,4.0,NaN,NaN,4.0,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
matriks_angka = matriks_user_film.fillna(0)

nilai_cosine = cosine_similarity(matriks_angka).copy()

np.fill_diagonal(nilai_cosine, 0)

similaritas_user = pd.DataFrame(
    nilai_cosine,
    index=matriks_user_film.index,
    columns=matriks_user_film.index
)

print("Cosine Similarity antar user berhasil dihitung.")

display(similaritas_user.iloc[:5, :5])

Cosine Similarity antar user berhasil dihitung.


userId,1,2,3,4,5
userId,,,,,
1,0.000000,0.033015,0.054166,0.173973,0.085789
2,0.033015,0.000000,0.000000,0.004563,0.019664
3,0.054166,0.000000,0.000000,0.002750,0.000000
4,0.173973,0.004563,0.002750,0.000000,0.098307
5,0.085789,0.019664,0.000000,0.098307,0.000000


In [9]:
user_contoh = 1

neighbor = similaritas_user.loc[user_contoh].sort_values(
    ascending=False
).head(10)

neighbor = neighbor.reset_index()
neighbor.columns = ["userId", "similarity"]

display(neighbor)

,userId,similarity
0,266,0.294945
1,57,0.293943
2,480,0.292427
3,469,0.284465
4,313,0.282261
5,368,0.278163
6,590,0.276499
7,39,0.275508
8,91,0.273268
9,577,0.271166


In [10]:
rata_user = train_data.groupby("userId")["rating"].mean()
rata_global = train_data["rating"].mean()

print("Rata-rata rating user berhasil dibuat")

Rata-rata rating user berhasil dibuat


In [11]:
def prediksi_rating(user_id, movie_id, k=10):
    if movie_id not in matriks_user_film.columns:
        return rata_user.get(user_id, rata_global)

    penilai_film = matriks_user_film[movie_id].dropna()

    nilai_sim = similaritas_user.loc[
        user_id,
        penilai_film.index
    ]

    nilai_sim = nilai_sim[nilai_sim > 0]
    nilai_sim = nilai_sim.sort_values(
        ascending=False
    ).head(k)

    if len(nilai_sim) == 0:
        return rata_user.get(user_id, rata_global)

    rating_neighbor = penilai_film.loc[nilai_sim.index]

    prediksi = (
        nilai_sim * rating_neighbor
    ).sum() / nilai_sim.sum()

    return float(prediksi)

In [12]:
contoh_test = test_data.iloc[0]

user_id = int(contoh_test["userId"])
movie_id = int(contoh_test["movieId"])
rating_asli = contoh_test["rating"]

rating_prediksi = prediksi_rating(
    user_id,
    movie_id,
    k=5
)

hasil_contoh = pd.DataFrame({
    "userId": [user_id],
    "movieId": [movie_id],
    "rating_asli": [rating_asli],
    "rating_prediksi": [round(rating_prediksi, 3)]
})

hasil_contoh = hasil_contoh.merge(
    referensi_film,
    on="movieId",
    how="left"
)

display(hasil_contoh)

,userId,movieId,rating_asli,rating_prediksi,title,genres
0,1,1224,5.0,3.995,Henry V (1989),Action|Drama|Romance|War


In [13]:
def prediksi_semua_data(k):
    hasil = test_data.copy()
    daftar_prediksi = []

    for _, baris in hasil.iterrows():
        user_id = int(baris["userId"])
        movie_id = int(baris["movieId"])

        prediksi = prediksi_rating(
            user_id,
            movie_id,
            k
        )

        daftar_prediksi.append(prediksi)

    hasil["rating_prediksi"] = daftar_prediksi
    hasil["K"] = k

    return hasil

In [15]:
hasil_k5 = prediksi_semua_data(5)

print("Prediksi untuk K = 5 selesai dibuat")
display(hasil_k5.head(10))

Prediksi untuk K = 5 selesai dibuat


,userId,movieId,rating,rating_prediksi,K
0,1,1224,5.0,3.995076,5
1,1,1920,4.0,2.902151,5
2,1,101,5.0,4.049801,5
3,1,296,3.0,4.589974,5
4,1,356,4.0,4.396194,5
5,1,1644,3.0,2.597172,5
6,1,2596,5.0,3.197186,5
7,1,1092,5.0,2.989005,5
8,1,661,5.0,3.484480,5
9,1,648,3.0,2.682543,5


In [16]:
hasil_k10 = prediksi_semua_data(10)

print("Prediksi untuk K = 10 selesai dibuat.")
display(hasil_k10.head(10))

Prediksi untuk K = 10 selesai dibuat.


,userId,movieId,rating,rating_prediksi,K
0,1,1224,5.0,4.088853,10
1,1,1920,4.0,2.859461,10
2,1,101,5.0,3.952616,10
3,1,296,3.0,4.690444,10
4,1,356,4.0,4.010214,10
5,1,1644,3.0,2.553276,10
6,1,2596,5.0,3.356439,10
7,1,1092,5.0,3.323639,10
8,1,661,5.0,3.439273,10
9,1,648,3.0,2.694200,10


In [17]:
hasil_k20 = prediksi_semua_data(20)

print("Prediksi untuk K = 20 selesai dibuat.")
display(hasil_k20.head(10))

Prediksi untuk K = 20 selesai dibuat.


,userId,movieId,rating,rating_prediksi,K
0,1,1224,5.0,3.947745,20
1,1,1920,4.0,2.890566,20
2,1,101,5.0,3.758106,20
3,1,296,3.0,4.646789,20
4,1,356,4.0,4.193735,20
5,1,1644,3.0,2.265033,20
6,1,2596,5.0,3.356439,20
7,1,1092,5.0,3.212193,20
8,1,661,5.0,3.356380,20
9,1,648,3.0,2.913905,20


In [18]:
def rekomendasi_film(user_id, k=10, jumlah_rekomendasi=10):
    film_sudah_dinilai = matriks_user_film.loc[
        user_id
    ].dropna().index

    neighbor = similaritas_user.loc[
        user_id
    ].sort_values(ascending=False).head(k)

    data_neighbor = matriks_user_film.loc[
        neighbor.index
    ]

    kandidat_film = data_neighbor.columns[
        data_neighbor.notna().any()
    ]

    hasil = []

    for movie_id in kandidat_film:
        if movie_id in film_sudah_dinilai:
            continue

        nilai_prediksi = prediksi_rating(
            user_id,
            movie_id,
            k
        )

        hasil.append([
            movie_id,
            nilai_prediksi
        ])

    hasil = pd.DataFrame(
        hasil,
        columns=["movieId", "rating_prediksi"]
    )

    hasil = hasil.merge(
        referensi_film,
        on="movieId",
        how="left"
    )

    hasil = hasil.sort_values(
        "rating_prediksi",
        ascending=False
    )

    return hasil.head(jumlah_rekomendasi)

In [19]:
rekomendasi_user1 = rekomendasi_film(
    user_id=1,
    k=10,
    jumlah_rekomendasi=10
)

display(
    rekomendasi_user1[
        ["title", "genres", "rating_prediksi"]
    ]
)

,title,genres,rating_prediksi
1023,Reform School Girls (1986),Action|Drama,5.000000
956,Phantasm II (1988),Action|Fantasy|Horror|Sci-Fi|Thriller,5.000000
206,"Godfather, The (1972)",Crime|Drama,4.899457
279,Paths of Glory (1957),Drama|War,4.715541
79,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,4.690444
84,"Shawshank Redemption, The (1994)",Crime|Drama,4.688973
152,Blade Runner (1982),Action|Sci-Fi|Thriller,4.615049
744,Fight Club (1999),Action|Crime|Drama|Thriller,4.604293
220,Casablanca (1942),Drama|Romance,4.597869
761,Yojimbo (1961),Action|Adventure,4.585844


In [20]:
hasil_k5.to_csv("hasil_prediksi_k5.csv", index=False)
hasil_k10.to_csv("hasil_prediksi_k10.csv", index=False)
hasil_k20.to_csv("hasil_prediksi_k20.csv", index=False)

rekomendasi_user1.to_csv(
    "rekomendasi_user1_k10.csv",
    index=False
)

print("Hasil modeling berhasil disimpan")

Hasil modeling berhasil disimpan
